In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit

source = spark.read.format("delta").load("/delta/bronze/customers")

target_path = "/delta/silver/customers_scd2"

if DeltaTable.isDeltaTable(spark, target_path):

    target = DeltaTable.Path(spark, target)

    target.alias("t").merge(
        source.alias("s"),
        "t.customer_id = s.customer_id AND t.is_current = true"
    ).whenMatchedUpdate(
        condition = "t.address <> s.address",
        set = {
            "is_current" = "false",
            "end_date" = "current_timestamp()"
        }
    ).whenNotMatchedInsert(
        values = {
            "customer_id":"s.customer_id",
            "address":"s.address",
            "is_current":"true",
            "start_date":"current_timestamp()",
            "end_date":"null"
        }
    )

else:
    source.withColumn("start_date", current_timestamp()) \
        .withColumn("end_date", lit(None))\
        .withColumn("is_current",lit(True))\
        .write.format("delta").save(target_path)
    